In [1]:
import os
import re
from pathlib import Path

import pandas as pd
import numpy as np
from PIL import Image

from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import StandardScaler


def _find_csv(filename):
    """
    Search for `filename` in typical Kaggle data folders and the current directory.
    Returns the first matching path as a string, or raises FileNotFoundError.
    """
    search_dirs = [
        Path("./data/plant-pathology-2020-fgvc7"),
        Path("./kaggle/input/plant-pathology-2020-fgvc7"),
        Path("./input/plant-pathology-2020-fgvc7"),
        Path("."),
    ]
    for d in search_dirs:
        candidate = d / filename
        if candidate.is_file():
            return str(candidate)
    for p in Path(".").rglob(filename):
        if p.is_file():
            return str(p)
    raise FileNotFoundError(f"Unable to locate {filename} in expected locations.")


def _find_images_dir():
    """
    Locate the directory that contains the image files (Train_*.jpg, Test_*.jpg).
    It is assumed to be a sibling 'images' folder next to the CSV files.
    """
    csv_path = Path(_find_csv("train.csv"))
    possible = [
        csv_path.parent / "images",
        csv_path.parent.parent / "images",
        Path("./data/plant-pathology-2020-fgvc7/images"),
        Path("./kaggle/input/plant-pathology-2020-fgvc7/images"),
        Path("./input/plant-pathology-2020-fgvc7/images"),
    ]
    for p in possible:
        if p.is_dir():
            return str(p)
    for p in Path(".").rglob("Train_*.jpg"):
        return str(p.parent)
    raise FileNotFoundError("Unable to locate images directory.")


def _numeric_id(img_id):
    """
    Extract the first integer appearing in the image_id string.
    If none found, return 0.
    """
    m = re.search(r"\d+", str(img_id))
    return int(m.group()) if m else 0


def _extract_features(image_path):
    """
    Load an image and compute lightweight visual statistics:
      - RGB channel mean & std (3 + 3)
      - HSV channel mean & std (3 + 3)
      - Aspect ratio (width / height) (1)
      - Normalized 16‑bin grayscale histogram (16)
    Returns a 30‑dimensional feature vector.
    """
    with Image.open(image_path) as img:
        orig_w, orig_h = img.size
        img = img.convert("RGB")
        img = img.resize((48, 48))

        arr_rgb = np.array(img) / 255.0  # (48,48,3)
        rgb_means = arr_rgb.mean(axis=(0, 1))
        rgb_stds = arr_rgb.std(axis=(0, 1))

        arr_hsv = np.array(img.convert("HSV")) / 255.0
        hsv_means = arr_hsv.mean(axis=(0, 1))
        hsv_stds = arr_hsv.std(axis=(0, 1))

        gray = img.convert("L")
        gray_arr = np.array(gray).ravel()
        hist, _ = np.histogram(gray_arr, bins=16, range=(0, 256), density=True)

        aspect_ratio = np.array([orig_w / orig_h if orig_h != 0 else 0.0])

    return np.concatenate(
        [rgb_means, rgb_stds, hsv_means, hsv_stds, aspect_ratio, hist]
    )  # (30,)


def compute_image_based_predictions():
    train_path = _find_csv("train.csv")
    test_path = _find_csv("test.csv")
    images_dir = _find_images_dir()

    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    label_cols = ["healthy", "multiple_diseases", "rust", "scab"]

    # ---------- Feature extraction for training ----------
    train_feats = []
    for img_id in train_df["image_id"]:
        img_file = os.path.join(images_dir, f"{img_id}.jpg")
        if not os.path.isfile(img_file):
            candidates = list(Path(images_dir).rglob(f"{img_id}.*"))
            img_file = str(candidates[0]) if candidates else None
        if img_file and os.path.isfile(img_file):
            feats = _extract_features(img_file)
        else:
            feats = np.zeros(30)
        train_feats.append(feats)

    X_color_train = np.vstack(train_feats)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_color_train)

    y_train = train_df[label_cols].values

    # ---------- More flexible logistic regression ----------
    base_clf = LogisticRegression(
        solver="lbfgs",
        # Remove class_weight='balanced' to let the model reflect true label frequencies
        class_weight=None,
        max_iter=2000,
        C=20.0,  # stronger fit (less regularisation)
        random_state=42,
        multi_class="ovr",
    )
    clf = OneVsRestClassifier(base_clf)
    clf.fit(X_train, y_train)

    # ---------- Feature extraction for test ----------
    test_feats = []
    for img_id in test_df["image_id"]:
        img_file = os.path.join(images_dir, f"{img_id}.jpg")
        if not os.path.isfile(img_file):
            candidates = list(Path(images_dir).rglob(f"{img_id}.*"))
            img_file = str(candidates[0]) if candidates else None
        if img_file and os.path.isfile(img_file):
            feats = _extract_features(img_file)
        else:
            feats = np.zeros(30)
        test_feats.append(feats)

    X_color_test = np.vstack(test_feats)
    X_test = scaler.transform(X_color_test)

    probs = clf.predict_proba(X_test)  # (n_test, n_labels)

    # ---------- Blend with global frequency baseline ----------
    global_means = train_df[label_cols].mean().values
    baseline = np.tile(global_means, (probs.shape[0], 1))

    # Trust the model more (97% model, 3% baseline)
    blended = 0.97 * probs + 0.03 * baseline

    pred_df = pd.DataFrame(blended, columns=label_cols)
    pred_df.insert(0, "image_id", test_df["image_id"])
    return pred_df




In [2]:
def write_submission(df, output_path="./submission.csv"):
    """
    Ensure the submission follows the exact column order required by Kaggle.
    """
    cols = ["image_id", "healthy", "multiple_diseases", "rust", "scab"]
    df = df[cols]
    df.to_csv(output_path, index=False)
    print(f"Submission written to {output_path}")




In [3]:
if __name__ == "__main__":
    submission_df = compute_image_based_predictions()
    print("First few predictions:")
    print(submission_df.head())
    write_submission(submission_df)

First few predictions:
  image_id   healthy  multiple_diseases      rust      scab
0   Test_0  0.198812           0.124873  0.470458  0.182571
1   Test_1  0.411544           0.048860  0.283060  0.225450
2   Test_2  0.147509           0.049083  0.353304  0.360556
3   Test_3  0.057619           0.001923  0.346189  0.693596
4   Test_4  0.072596           0.010206  0.837434  0.089825
Submission written to ./submission.csv
